# Project ATLAS: Google Colab GPU Campaign Runner

**Associative Topological LATent Skills (ATLAS)**: Reuses the frozen T2 world model backbone, self-learning visual skills, topological navigation, and epistemic curiosity around it.

### Storage Contract:
- **Main Google Drive folder**: `/content/drive/MyDrive/learn-a-world` (shared dataset & pre-trained checkpoints)
- **Input sequence data**: `/content/drive/MyDrive/learn-a-world/data/sequences/*.npz`
- **Frozen backbone checkpoint**: `/content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt`
- **Dedicated ATLAS subfolder**: `/content/drive/MyDrive/learn-a-world/atlas/`
  - `atlas/shards/`: newly collected shards and cached `.latents.npz`
  - `atlas/runs/<run_id>/`: checkpoints (`state.pt`), `goal_bank`, `lift.npz`, `topomap.npz`, `train_log.jsonl`
  - `atlas/results/`: official benchmark screening reports (`atlas_screen_<tag>.json`)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/learn-a-world")
ATLAS_DIR = DRIVE_DIR / "atlas"

# Create dedicated ATLAS subfolders on Drive
for sub in ["shards", "runs", "checkpoints", "results"]:
    (ATLAS_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Google Drive mounted successfully.")
print(f"Main Drive root: {DRIVE_DIR}")
print(f"ATLAS dedicated folder: {ATLAS_DIR}")

## 2. Sync Repository

In [ ]:
%cd /content
import os

if not os.path.exists("learn-a-world"):
    !git clone https://github.com/sahiltolani30/learn-a-world.git

%cd /content/learn-a-world
!git fetch origin
!git checkout main
!git pull origin main
!git log -1 --oneline

## 3. Environment Setup & Hardware Verification

In [ ]:
%cd /content/learn-a-world
!bash atlas/colab/setup_atlas.sh

## 4. Run Unit Tests & Local Smoke Verification

In [ ]:
%cd /content/learn-a-world
# Run all 26 ATLAS unit tests
!python3 -m pytest atlas/tests -v

# Run end-to-end 2-episode smoke test
!python3 -m atlas.smoke

## 5. Check Drive Checkpoint and Sequences Availability

In [ ]:
import glob
from pathlib import Path

drive_ckpt = Path("/content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt")
drive_seqs = sorted(glob.glob("/content/drive/MyDrive/learn-a-world/data/sequences/*.npz"))

print(f"Champion Checkpoint exists on Drive: {drive_ckpt.exists()}")
print(f"Found {len(drive_seqs)} sequence shards on Drive for bootstrapping.")

## 6. Phase 1: Offline Bootstrap on Drive Sequences
Reads pre-collected sequence shards from Drive, caches latents into `atlas/shards/`, trains OptionNet, ReachabilityNet, EpistemicNet, ContinueNet, and mines the initial `GoalBank` and `PredecessorLift`.

In [ ]:
%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase bootstrap \
  --iters 10 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

## 7. Phase 2: Autonomous Self-Learn Loop
Executes iterative cycles: `[Collect episodes -> Shard -> Cache latents -> Train heads -> Mine GoalBank -> Save state.pt]`.
Gracefully stops before the `--minutes` budget expires and saves state atomically directly to Drive.

In [ ]:
%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase selflearn \
  --iters 40 \
  --minutes 180 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

## 8. Phase 3: Locked 50-Episode Benchmark Screening
Evaluates the trained ATLAS agent under strict locked protocol: seeds 2050–2099, 50 episodes, max 1000 steps, zero exploration (epsilon=0, beta=0). Results are written to `results/` and `/content/drive/MyDrive/learn-a-world/atlas/results/`.

In [ ]:
%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase screen \
  --tag atlas_v1 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

## 9. Inspect Screening Results & Achievements

In [ ]:
import json
from pathlib import Path

res_file = Path("/content/drive/MyDrive/learn-a-world/atlas/results/atlas_screen_atlas_v1.json")
if not res_file.exists():
    res_file = Path("results/atlas_screen_atlas_v1.json")

if res_file.exists():
    with open(res_file) as f:
        data = json.load(f)
    print(f"Official Crafter Score: {data['crafter_score']:.2f}%")
    print(f"Mean Survival: {data['mean_survival']:.1f} steps | Mean Reward: {data['mean_reward']:.2f}")
    print(f"Telemetry: {data['per_mode_telemetry']}")
    print("\nKey Milestone Unlocks (% of episodes):")
    for k in ["collect_wood", "place_table", "make_wood_pickaxe", "collect_stone", "make_stone_pickaxe", "collect_coal", "place_furnace", "collect_iron"]:
        print(f"  {k:20s}: {data['unlock_percentages'].get(k, 0.0):.1f}%")
else:
    print("No screening report found yet.")